# 02 · Preparação dos dados
**CRISP-DM:** Preparação dos Dados

Gera a tabela **recurso × janela** que alimenta todos os modelos:
1. carga com conversão UTC → Brasília;
2. consolidação das chaves repetidas;
3. agregação por janela (padrão 1h) com os atributos de comportamento e de contexto;
4. divisão temporal.

| Grupo | Atributo | Definição |
|---|---|---|
| Volume | `log_vol` | log(1 + transações na janela) |
| Erros | `tx_4xx`, `tx_5xx` | transações 4xx (5xx) / total |
| Latência | `log_lat_2xx` | log(1 + latência média das respostas 2xx, ponderada pelo volume) |
| Clientes | `log_n_cli` | log(1 + nº de clientes distintos) |
| | `top1_vol` | participação do maior cliente no volume |
| | `frac_cli_4xx`, `frac_cli_5xx` | fração dos clientes ativos com ao menos um 4xx (5xx) |
| | `conc_4xx`, `conc_5xx` | participação dos 2 clientes com mais erros no total de erros (usado só no lado provável) |
| Contexto | `hora`, `tipo_dia` | hora (Brasília) e útil / sábado / domingo-feriado |

In [1]:
%run ./00_funcoes.ipynb

00_funcoes carregado · raiz: /home/wsl/gh/mestrado/claude9


In [2]:
# Parâmetros
JANELAS_GERAR = JANELAS_SENSIBILIDADE  # gera todas as janelas usadas no experimento de sensibilidade

In [3]:
brutos = consolidar(carregar())
print(f"registros consolidados: {len(brutos):,}")
brutos.to_parquet(RESULTADOS / "02_brutos_consolidados.parquet", index=False)

registros consolidados: 2,283,149


In [4]:
limites = {}
for jn in JANELAS_GERAR:
    jan = agregar(brutos, jn)
    jan.to_parquet(RESULTADOS / f"02_janelas_{jn}.parquet", index=False)
    limites[jn] = limites_recursos(brutos, jn)
    print(f"{jn:>6}: {len(jan):,} janelas · {jan['recurso'].nunique()} recursos")

jan = pd.read_parquet(RESULTADOS / f"02_janelas_{JANELA}.parquet")
limites[JANELA].to_csv(RESULTADOS / "02_limites_recursos.csv")

 15min: 42,783 janelas · 12 recursos


 30min: 21,391 janelas · 12 recursos


    1h: 10,695 janelas · 12 recursos


    4h: 2,672 janelas · 12 recursos


## Divisão temporal (horário de Brasília)
O limiar τ de cada recurso é calibrado por validação cruzada com 4 dobras semanais dentro de agosto. O teste (01–07/09) não participa de nenhum ajuste.

In [5]:
semanas = [(pd.Timestamp("2026-08-01"), pd.Timestamp("2026-08-08")), (pd.Timestamp("2026-08-08"), pd.Timestamp("2026-08-15")),
           (pd.Timestamp("2026-08-15"), pd.Timestamp("2026-08-22")), (pd.Timestamp("2026-08-22"), pd.Timestamp("2026-09-01"))]
divisao = pd.DataFrame({
    "conjunto": [f"Treino · dobra {i + 1} da validação cruzada" for i in range(4)] + ["Treino (modelo final)", "Teste (falhas injetadas)"],
    "inicio": [a for a, _ in semanas] + [TREINO[0], TESTE[0]],
    "fim_exclusivo": [b for _, b in semanas] + [TREINO[1], TESTE[1]],
})
divisao["janelas_1h"] = [len(recortar(jan, (a, b))) for a, b in zip(divisao["inicio"], divisao["fim_exclusivo"])]
divisao

,conjunto,inicio,fim_exclusivo,janelas_1h
0,Treino · dobra 1 da validação cruzada,2026-08-01,2026-08-08,2016
1,Treino · dobra 2 da validação cruzada,2026-08-08,2026-08-15,2016
2,Treino · dobra 3 da validação cruzada,2026-08-15,2026-08-22,2016
3,Treino · dobra 4 da validação cruzada,2026-08-22,2026-09-01,2880
4,Treino (modelo final),2026-08-01,2026-09-01,8928
5,Teste (falhas injetadas),2026-09-01,2026-09-08,1564


## Amostras por contexto no treino
Com janelas de 1h, cada slot recurso × tipo de dia × hora tem ~21 amostras em dias úteis e ~5 aos sábados e domingos.

In [6]:
tr = recortar(jan, TREINO)
amostras = tr.groupby(["recurso", "tipo_dia", "hora"]).size().groupby("tipo_dia").describe()[["min", "50%", "max"]]
amostras

,min,50%,max
tipo_dia,,,
domingo_feriado,5.0,5.0,5.0
sabado,5.0,5.0,5.0
util,21.0,21.0,21.0


In [7]:
jan[ATRIBUTOS + ["vol", "lat_2xx", "n_cli"]].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
log_vol,10695.0,7.140,1.349,2.485,6.301,7.039,7.970,11.700
tx_4xx,10695.0,0.281,0.377,0.000,0.001,0.015,0.571,1.000
tx_5xx,10695.0,0.003,0.023,0.000,0.000,0.000,0.000,0.821
log_lat_2xx,10187.0,4.146,0.528,2.851,3.752,4.040,4.481,8.051
log_n_cli,10695.0,2.623,1.094,0.693,1.792,2.565,3.434,6.205
top1_vol,10695.0,0.588,0.246,0.046,0.405,0.557,0.783,1.000
frac_cli_4xx,10695.0,0.300,0.349,0.000,0.020,0.143,0.455,1.000
frac_cli_5xx,10695.0,0.081,0.242,0.000,0.000,0.000,0.020,1.000
vol,10695.0,3355.371,8348.082,11.000,544.000,1139.000,2890.500,120553.000
lat_2xx,10187.0,74.945,98.977,16.304,41.598,55.829,87.348,3137.215


In [8]:
print("janelas sem nenhuma resposta 2xx (latência 2xx indefinida):", int(jan["lat_2xx"].isna().sum()))
print("janelas com volume zero:", int((jan["vol"] == 0).sum()))

janelas sem nenhuma resposta 2xx (latência 2xx indefinida): 508
janelas com volume zero: 0
